# EEG · 실제 공개 데이터로 눈 깜빡임 찾기

합성 데이터 예제(`eeg_synthetic_example.ipynb`)에서 쓴 방법을 **실제 사람의 기록**에 그대로 적용합니다.

- 데이터: PhysioNet **EEG Motor Movement/Imagery Dataset**(EEGBCI). 109명이 64채널 뇌파 모자를 쓰고 기록했고, 1초에 160번 측정했습니다.
- 이 노트북은 그중 **run 1**(눈을 뜨고 가만히 있는 1분 기록)만 씁니다.
- 실제 기록에는 정답 파일이 없습니다. 찾은 것이 깜빡임인지는 파형과 두피 분포를 보고 스스로 판단해야 합니다.

이용 조건: Open Data Commons Attribution License v1.0. 출처는 https://physionet.org/content/eegmmidb/1.0.0/ 에 있습니다. 결과를 공유할 때 그 페이지가 요구하는 인용을 함께 적어 주세요.

## 0. 설정

받을 위치와 사람 번호를 정합니다. 받을 위치는 노트북 옆 `data_real/` 폴더가 기본값입니다. 다른 곳에 받고 싶으면 `DATA_DIR`만 바꾸세요. 사람 번호는 1–109 중에서 고릅니다.

In [ ]:
DATA_DIR = "data_real"        # 데이터를 받을 폴더. 바꿔도 됩니다.
SUBJECTS = [1, 2, 3, 4, 5, 6]  # 받을 사람 번호 (1–109)
THRESHOLD = 80                # µV. 합성 예제와 같은 기준값

## 1. 데이터 받기

`mne.datasets.eegbci.load_data`가 PhysioNet에서 파일을 받습니다. 한 사람의 run 1은 EDF 파일 하나입니다. 이미 받은 파일이 있으면 다시 받지 않습니다.

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
from mne.datasets import eegbci
from scipy.signal import find_peaks

Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
t0 = time.perf_counter()
files = eegbci.load_data(SUBJECTS, runs=[1], path=DATA_DIR, update_path=False, verbose="error")
seconds = time.perf_counter() - t0
size_mb = sum(Path(f).stat().st_size for f in files) / 1e6
print(f"파일 {len(files)}개, {size_mb:.1f} MB, {seconds:.1f}초")

## 2. 채널 이름과 단위 맞추기

EEGBCI 파일에는 64채널이 있고, 이름이 `Fp1.`, `Cz..`처럼 점으로 끝납니다. 합성 데이터와 비교하려면 세 가지를 맞춥니다.

1. **이름**: 점을 지우고 대소문자를 10-20 표준 이름에 맞춥니다.
2. **채널**: 64채널 중 합성 데이터와 같은 19채널(국제 10-20 배치)만 고릅니다.
3. **기준과 단위**: 19채널 평균을 기준으로 바꾸고(합성 데이터도 평균 기준), 볼트(V)를 마이크로볼트(µV)로 바꿉니다.

In [ ]:
CHANNELS = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T7", "C3", "Cz",
            "C4", "T8", "P7", "P3", "Pz", "P4", "P8", "O1", "O2"]

# 표준 전극 위치. MNE 1.13부터 이름이 colin27_1020으로 바뀌었습니다.
MONTAGE = "colin27_1020" if "colin27_1020" in mne.channels.get_builtin_montages() else "standard_1020"

def load_subject(path):
    raw = mne.io.read_raw_edf(path, preload=True, verbose="error")
    lookup = {name.upper(): name for name in CHANNELS}
    mapping = {ch: lookup[ch.strip(".").upper()] for ch in raw.ch_names
               if ch.strip(".").upper() in lookup}
    raw.rename_channels(mapping)
    raw.pick(CHANNELS)                                   # 19채널만, 합성 데이터와 같은 순서
    raw.set_montage(MONTAGE)                             # 두피 지도를 그리려면 전극 위치가 필요
    raw.set_eeg_reference("average", verbose="error")    # 19채널 평균 기준
    return raw

raws = [load_subject(f) for f in files]
raw = raws[0]
print("원래 이름 예:", mne.io.read_raw_edf(files[0], verbose="error").ch_names[:6])
print("바꾼 이름:", raw.ch_names)
print(f"초당 {raw.info['sfreq']:.0f}번 측정, 길이 {raw.times[-1]:.0f}초")

## 3. 신호 그려 보기

첫 번째 사람의 처음 20초를 그립니다. 합성 예제의 `plot_channels`와 같은 함수입니다.

In [ ]:
def plot_channels(raw, start, length, channels=CHANNELS, spacing=150, marks=()):
    data = raw.get_data() * 1e6          # V → µV
    sfreq, chs = raw.info["sfreq"], raw.ch_names
    s, e = int(start * sfreq), int((start + length) * sfreq)
    t = np.arange(data.shape[1]) / sfreq
    fig, ax = plt.subplots(figsize=(13, 0.45 * len(channels) + 1))
    for k, ch in enumerate(channels):
        ax.plot(t[s:e], data[chs.index(ch), s:e] - spacing * k, lw=0.7, color="k")
    for m in marks:
        if start <= m <= start + length:
            ax.axvline(m, color="r", alpha=0.3)
    ax.set_yticks([-spacing * k for k in range(len(channels))], channels)
    ax.set_xlabel("time (s)")
    ax.set_title(f"vertical spacing = {spacing} µV")
    return ax

plot_channels(raw, 0, 20);

**생각해 보기:** 합성 데이터의 그림과 무엇이 다른가요? 깜빡임처럼 보이는 파형이 있나요? 신호가 천천히 위아래로 떠다니는 채널이 있나요?

## 4. 같은 방법으로 큰 파형 찾기

합성 예제 2단계와 똑같이 합니다. 이마 채널(Fp1, Fp2)의 평균을 10 Hz 저역 통과하고, `THRESHOLD`보다 높이 솟은 정점을 0.3초 이상 간격으로 찾습니다. 사람마다 분당 횟수와 정점 크기(중앙값)를 출력합니다.

**배경 크기**는 정점을 찾는 신호가 평소에 흔들리는 폭입니다(중앙값에서 벗어난 정도의 중앙값 × 1.4826, 정규분포의 표준편차에 맞춘 값). 기준값이 배경보다 충분히 크지 않으면 깜빡임이 아닌 것도 셉니다.

In [ ]:
def frontal_slow(raw):
    data = raw.get_data() * 1e6
    frontal = data[[raw.ch_names.index("Fp1"), raw.ch_names.index("Fp2")]].mean(axis=0)
    return mne.filter.filter_data(frontal, raw.info["sfreq"], None, 10.0, verbose="error")

def detect(raw, threshold=THRESHOLD):
    sfreq = raw.info["sfreq"]
    slow = frontal_slow(raw)
    peaks, props = find_peaks(slow, height=threshold, distance=int(0.3 * sfreq))
    return peaks, props["peak_heights"]

def robust_sd(x):
    return np.median(np.abs(x - np.median(x))) * 1.4826

results = []
for subject, r in zip(SUBJECTS, raws):
    peaks, heights = detect(r)
    minutes = r.n_times / r.info["sfreq"] / 60
    results.append(dict(subject=subject, minutes=minutes, n=len(peaks), rate=len(peaks) / minutes,
                        height=np.median(heights) if len(peaks) else np.nan,
                        background=robust_sd(frontal_slow(r))))
    print(f"S{subject:03d}: {minutes:.2f}분, 정점 {len(peaks)}개, 분당 {len(peaks) / minutes:.1f}회, "
          f"정점 크기 중앙값 {results[-1]['height']:.0f} µV, 배경 크기 {results[-1]['background']:.0f} µV")
total_n = sum(x["n"] for x in results)
total_min = sum(x["minutes"] for x in results)
print(f"\n{len(results)}명 합계: 정점 {total_n}개 / {total_min:.1f}분 = 분당 {total_n / total_min:.1f}회")

모든 사람의 이마 신호와 찾은 정점을 한 번에 그립니다. 빨간 점이 깜빡임처럼 생겼는지 눈으로 확인하세요.

In [ ]:
fig, axes = plt.subplots(len(raws), 1, figsize=(13, 1.6 * len(raws)), sharex=True)
for ax, subject, r in zip(np.atleast_1d(axes), SUBJECTS, raws):
    slow = frontal_slow(r)
    t = np.arange(slow.size) / r.info["sfreq"]
    peaks, _ = detect(r)
    ax.plot(t, slow, lw=0.6, color="k")
    ax.plot(t[peaks], slow[peaks], "r.", ms=6)
    ax.axhline(THRESHOLD, color="r", lw=0.5, ls="--")
    ax.set_ylabel(f"S{subject:03d}\nµV")
ax.set_xlabel("time (s)");

## 5. 기준값을 바꾸면?

합성 데이터에서는 기준값을 50이나 120으로 바꿔도 틀린 것이 생기지 않았습니다. 실제 기록에서는 어떤지 사람별로 셉니다.

In [ ]:
levels = [50, 80, 120, 200]
print("사람   " + "".join(f"{lv:>7} µV" for lv in levels))
for subject, r in zip(SUBJECTS, raws):
    print(f"S{subject:03d}  " + "".join(f"{len(detect(r, lv)[0]):>10}" for lv in levels))

## 6. 파형이 어디서 왔을까? 두피 분포

찾은 정점마다 모든 채널에서 "정점 값 − 0.15초 전 값"을 구해 평균합니다(합성 예제 3단계와 같은 방법). 기록 맨 앞 0.15초 안의 정점은 뺍니다. 사람마다 평균을 낸 뒤 다시 사람끼리 평균합니다.

In [ ]:
maps = []
for r in raws:
    data, sfreq = r.get_data() * 1e6, r.info["sfreq"]
    lag = int(0.15 * sfreq)
    peaks = [p for p in detect(r)[0] if p >= lag]
    if peaks:
        maps.append(np.mean([data[:, p] - data[:, p - lag] for p in peaks], axis=0))
amplitude = np.mean(maps, axis=0)
fig, ax = plt.subplots(figsize=(4, 4))
mne.viz.plot_topomap(amplitude, raw.info, axes=ax, show=False, cmap="RdBu_r")
ax.set_title(f"mean change at detected peaks, {len(maps)} people (µV)");
for ch in ["Fp1", "Fz", "Cz", "Pz", "O1"]:
    print(f"{ch}: {amplitude[CHANNELS.index(ch)]:+.0f} µV")

## 7. 합성 데이터와 비교

같은 폴더에 합성 데이터(`data/eeg_raw.fif`)를 만들어 두었다면 같은 함수로 숫자를 구해 나란히 놓습니다. 없으면 `python generate_synthetic_eeg.py`를 먼저 실행하세요.

In [ ]:
synth_path = Path("data/eeg_raw.fif")
if synth_path.exists():
    synth = mne.io.read_raw_fif(synth_path, preload=True, verbose="error")
    peaks, heights = detect(synth)
    minutes = synth.n_times / synth.info["sfreq"] / 60
    print(f"합성: 분당 {len(peaks) / minutes:.1f}회, 정점 크기 중앙값 {np.median(heights):.0f} µV, "
          f"배경 크기 {robust_sd(frontal_slow(synth)):.0f} µV")
else:
    print("data/eeg_raw.fif가 없습니다. python generate_synthetic_eeg.py 를 먼저 실행하세요.")
rates = [x["rate"] for x in results]
print(f"실제: 분당 {min(rates):.1f}–{max(rates):.1f}회 (사람별), "
      f"정점 크기 중앙값 {np.nanmedian([x['height'] for x in results]):.0f} µV, "
      f"배경 크기 {np.median([x['background'] for x in results]):.0f} µV (사람별 값의 중앙값)")

## 설명해 보세요

1. 합성 데이터에서 쓴 방법이 실제 기록에서도 깜빡임을 찾았나요? 4단계 그림에서 깜빡임이 아닌 것 같은 정점을 골라 이유를 적어 보세요.
2. 깜빡임 빈도와 크기를 합성 데이터와 비교하세요. 사람마다 얼마나 다른가요?
3. 실제 기록에서만 생기는 어려움을 적어 보세요. 힌트: 정답 파일이 없다, 신호가 천천히 떠다닌다(기저선 표류), 사람마다 배경 크기가 다르다, 근육 잡음, 측정 속도가 다르다(160 Hz와 250 Hz), 기록 맨 끝 1초 가까이가 0으로 채워져 있다(4단계 그림 오른쪽 끝). 기준값 하나를 모든 사람에게 쓰는 대신 사람마다 기준을 정하려면 어떻게 하면 좋을까요?

데이터 출처: Schalk, G. (2009). EEG Motor Movement/Imagery Dataset (version 1.0.0). PhysioNet. https://doi.org/10.13026/C28G6P